# LOCUS — ARC-AGI Training Notebook

This notebook runs the LOCUS companion agent against ARC-AGI games using the Anthropic API.
LOCUS reads `companion_arcprize.md` as its system prompt and picks actions based on its game-mechanic knowledge graph.

**Before running:** add `ANTHROPIC_API_KEY` and `ARC_API_KEY` under  
*notebook → Settings → Add-ons → Secrets* then toggle each secret on for this notebook.

In [ ]:
# Install dependencies
!pip install anthropic arc-agi -q

In [ ]:
# Inject Kaggle secrets into environment
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")
os.environ["ARC_API_KEY"]       = secrets.get_secret("ARC_API_KEY")

In [ ]:
# Download kaggle_agent.py from the companion_arc repo
!wget -q https://raw.githubusercontent.com/antfriend/companion_arc/main/kaggle_agent.py

In [ ]:
# Setup — fetches companion_arcprize.md and initialises the Anthropic client
from kaggle_agent import setup, run_training_attempt

client, companion = setup()

In [ ]:
# Run one training attempt in practice mode
# Change game_id to the target task (e.g. "ls20")
# Set max_steps high enough to cover the level sequence
result = run_training_attempt(
    game_id="ls20",
    client=client,
    companion_text=companion,
    max_steps=300,
    competition_mode=False,   # flip to True for leaderboard submission
    verbose=True,
)

print("\n--- Result ---")
print(f"steps:            {result['steps']}")
print(f"levels_completed: {result['levels_completed']}")
print(f"final_state:      {result['final_state']}")
print(f"scorecard:\n{result['scorecard']}")
print(f"\nLOCUS session log: {result['locus_session_log']}")
print("Open that file and apply LOCUS's suggested updates to companion_arcprize.md")

## After a run: updating companion_arcprize.md

`run_training_attempt` writes a `locus_ls20_session.txt` file containing every LOCUS exchange from the session — STATUS, each action choice, level LOG entries, revision cycle responses, and the final game-end LOG.

**To apply the learnings:**

1. Open `locus_ls20_session.txt` in VS Code
2. Open `companion_arcprize.md` alongside it
3. In Claude Code, paste the LOG entries from the session file:
   ```
   @LOCUS LOG level 1 complete — 17 actions — [mechanic observation from LOCUS]
   ```
4. Claude Code will write the updated records into `companion_arcprize.md`

**What LOCUS logs during a run:**

| Event | LOCUS call | Purpose |
|---|---|---|
| Session start | `FOCUS lat-10lon10` | Increment sal on game state record |
| Session start | `STATUS` | EPS scan — surface high-strain beliefs |
| Each step | state + frame | Action decision with mechanic reasoning |
| Level win | `LOG level N complete` | Record step count + frame state |
| Level win | revision cycle prompt | Phases 1-3: notice → encounter → revise |
| Game end | `LOG win/game_over` | Final outcome with frame summary |

The revision cycle's **Phase 4** (validate) fires automatically the next time you log a level outcome — the next run confirms or refutes whatever was revised.

## Switching to competition mode

When ready for a leaderboard submission, change `competition_mode=True`.  
Competition constraints: API-only, no local resets, single environment, score is final on `get_scorecard()`.